In [52]:
def border_line(length=64):
    print("=-" * int(length / 2), end="=\n")

In [53]:
border_line(58)
print("Análise de anúncios de aluguel de imóveis de curta duração.")
border_line(58)
print("\nA base de dados é um CSV de 17 colunas e 12'044 entradas.\n")
print(
    "Os campos textuais em algumas entradas não trataram devidamente as vírgulas, o que separou uma coluna em múltiplas."
)
print("Os dados foram importados no LibreOffice para encontrar essas distorções.\n")
print('Coluna "Nome do anúncio": 8 distorções.')
print(
    'As linhas foram ordenadas pela coluna seguinte, "ID do anunciante". Aquelas com valores textuais ficaram ao final.\n'
)

print('Coluna "Licença": 6 distorções.')
print(
    "Como é a última coluna esperada, qualquer dado que se apresentava após si foi identificado."
)

=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
Análise de anúncios de aluguel de imóveis de curta duração.
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=

A base de dados é um CSV de 17 colunas e 12'044 entradas.

Os campos textuais em algumas entradas não trataram devidamente as vírgulas, o que separou uma coluna em múltiplas.
Os dados foram importados no LibreOffice para encontrar essas distorções.

Coluna "Nome do anúncio": 8 distorções.
As linhas foram ordenadas pela coluna seguinte, "ID do anunciante". Aquelas com valores textuais ficaram ao final.

Coluna "Licença": 6 distorções.
Como é a última coluna esperada, qualquer dado que se apresentava após si foi identificado.


In [54]:
import textwrap
from typing import Literal

import pandas as pd

indentation = "    "


def print_numerical_range(
    series: pd.Series,
):
    min_value, max_value = series.min(), series.max()
    if hasattr(min_value, "strftime") and hasattr(max_value, "strftime"):
        min_text = min_value.strftime("%Y-%m-%d")
        max_text = max_value.strftime("%Y-%m-%d")
    else:
        min_text = str(min_value)
        max_text = str(max_value)
    text = f"Intervalo numérico: [{min_text}, {max_text}]"
    print(textwrap.indent(text, indentation))


def column_description(
    name: str,
    series: pd.Series,
    type_description: str,
    description: str,
    column_type: Literal["categorical", "numerical", "date", "other"] = "other",
):

    print(name)

    empty_amount = series.isnull().sum()
    total_amount = series.count()
    text = f"Identificador: {series.name}\nTipo: {type_description}\n{description}\nVazios: {empty_amount} de {total_amount} ({(empty_amount / total_amount):.1%})"
    print(
        textwrap.indent(
            text,
            indentation,
        )
    )

    if column_type == "categorical":
        values = sorted(series.dropna().unique().tolist())
        text = f"Valores possíveis: {len(values)} {values!s}"
        print(textwrap.indent(text, indentation))

    elif column_type == "date":
        print_numerical_range(series)

    elif column_type == "numerical":
        print_numerical_range(series)
        text = f"Média: {series.mean()}\nMediana: {series.median()}\nDesvio padrão: {series.std()}"
        print(textwrap.indent(text, indentation))

In [55]:
dataset = pd.read_csv("data/1_Porto.csv")
dataset["last_review"] = pd.to_datetime(
    dataset["last_review"], errors="coerce", format="%Y-%m-%d"
)

border_line(6)
print("Colunas")
border_line(6)
print()

column_description(
    name="ID do anúncio",
    series=dataset.id,
    type_description="Textual (sequência de dígitos)",
    description="Identificador interno dado pelo Airbnb para esta entrada.",
    column_type="other",
)
# Creio ser irrelevante.
print()

column_description(
    name="Nome do anúncio",
    series=dataset.name,
    type_description="Textual",
    description="Nome dado pelos anunciantes a este anúncio.",
    column_type="other",
)
# Creio ser irrelevante.
print()

column_description(
    name="ID do anunciante",
    series=dataset.host_id,
    type_description="Textual (sequência de dígitos)",
    description="Identificador interno dado pelo Airbnb para a conta do anunciante que registrou esta entrada.",
    column_type="other",
)
# Creio ser irrelevante.
print()

column_description(
    name="Nome do anunciante",
    series=dataset.host_name,
    type_description="Textual",
    description="Nome do anunciante que registrou esta entrada. Mesmo quando há mais de uma pessoa, eles são responsáveis pela mesma conta no Airbnb.",
    column_type="other",
)
# Creio ser irrelevante.
print()

column_description(
    name="Município do imóvel",
    series=dataset.neighbourhood_group,
    type_description="Categórico, não ordenado",
    description="Município em que o imóvel anunciado está localizado. Obtido por meio da localização geográfica.",
    column_type="categorical",
)
dataset.neighbourhood_group = pd.Categorical(
    dataset.neighbourhood_group,
    categories=[
        "AROUCA",
        "ESPINHO",
        "GONDOMAR",
        "MAIA",
        "MATOSINHOS",
        "OLIVEIRA DE AZEMÉIS",
        "PAREDES",
        "PORTO",
        "PÓVOA DE VARZIM",
        "SANTA MARIA DA FEIRA",
        "SANTO TIRSO",
        "SÃO JOÃO DA MADEIRA",
        "TROFA",
        "VALE DE CAMBRA",
        "VALONGO",
        "VILA DO CONDE",
        "VILA NOVA DE GAIA",
    ],
    ordered=False,
)
# Quais municípios serão mais baratos?
print()

column_description(
    name="Freguesia do imóvel",
    series=dataset.neighbourhood,
    type_description="Categórico, não ordenado",
    description="Freguesia em que o imóvel anunciado está localizado. Está dentro de apenas um município.",
    column_type="categorical",
)
dataset.neighbourhood = pd.Categorical(
    dataset.neighbourhood,
    categories=[
        "AVer-o-Mar, Amorim e Terroso",
        "Agrela",
        "Aguiar de Sousa",
        "Aguçadoura e Navais",
        "Aldoar, Foz do Douro e Nevogilde",
        "Alfena",
        "Alvarelhos e Guidões",
        "Alvarenga",
        "Anta e Guetim",
        "Arcozelo",
        "Areias, Sequeiró, Lama e Palmeira",
        "Argoncilhe",
        "Arouca e Burgo",
        "Arrifana",
        "Arões",
        "Aveleda",
        "Avintes",
        "Azurara",
        "Baguim do Monte (Rio Tinto)",
        "Bagunte, Ferreiró, Outeiro Maior e Parada",
        "Balazar",
        "Baltar",
        "Bonfim",
        "Bougado (São Martinho e Santiago)",
        "Cabreiros e Albergaria da Serra",
        "Caldas de São Jorge e de Pigeiros",
        "Campanhã",
        "Campo e Sobrado",
        "Canedo, Vale e Vila Maior",
        "Canelas",
        "Canelas e Espiunca",
        "Canidelo",
        "Carregosa",
        "Carreira e Refojos de Riba de Ave",
        "Castêlo da Maia",
        "Cedofeita, Ildefonso, Sé, Miragaia, Nicolau, Vitória",
        "Cepelos",
        "Cesar",
        "Cete",
        "Chave",
        "Cidade da Maia",
        "Coronado (São Romão e São Mamede)",
        "Covelas",
        "Covelo de Paivó e Janarde",
        "Cristelo",
        "Custóias, Leça do Balio e Guifões",
        "Duas Igrejas",
        "Ermesinde",
        "Escariz",
        "Espinho",
        "Estela",
        "Fajões",
        "Fermedo",
        "Fiães",
        "Fornelo e Vairão",
        "Fornos",
        "Foz do Sousa e Covelo",
        "Fânzeres e São Pedro da Cova",
        "Gandra",
        "Gião",
        "Gondomar (São Cosme), Valbom e Jovim",
        "Grijó e Sermonde",
        "Guilhabreu",
        "Gulpilhares e Valadares",
        "Junqueira",
        "Labruge",
        "Laundos",
        "Lobão, Gião, Louredo e Guisande",
        "Lomba",
        "Lordelo do Ouro e Massarelos",
        "Louredo",
        "Lourosa",
        "Macieira da Maia",
        "Macieira de Cambra",
        "Macieira de Sarnes",
        "Madalena",
        "Mafamude e Vilar do Paraíso",
        "Malta e Canidelo",
        "Mansores",
        "Matosinhos e Leça da Palmeira",
        "Melres e Medas",
        "Milheirós",
        "Milheirós de Poiares",
        "Mindelo",
        "Moldes",
        "Monte Córdova",
        "Moreira",
        "Mozelos",
        "Muro",
        "Negrelos (São Tomé)",
        "Nogueira da Regedoura",
        "Nogueira do Cravo e Pindelo",
        "Nogueira e Silva Escura",
        "O. Azeméis, Riba-Ul, Ul, Macinhata da Seixa, Madail",
        "Oliveira do Douro",
        "Ossela",
        "Parada de Todeia",
        "Paramos",
        "Paranhos",
        "Paredes",
        "Paços de Brandão",
        "Pedroso e Seixezelo",
        "Pedrouços",
        "Perafita, Lavra e Santa Cruz do Bispo",
        "Pinheiro da Bemposta, Travanca e Palmaz",
        "Póvoa de Varzim, Beiriz e Argivai",
        "Ramalde",
        "Rates",
        "Rebordosa",
        "Reguenga",
        "Retorta e Tougues",
        "Rio Mau e Arcos",
        "Rio Tinto",
        "Roge",
        "Romariz",
        "Roriz",
        "Rossas",
        "Sandim, Olival, Lever e Crestuma",
        "Sanguedo",
        "Santa Eulália",
        "Santa Maria da Feira, Travanca, Sanfins e Espargo",
        "Santa Maria de Lamas",
        "Santa Marinha e São Pedro da Afurada",
        "Serzedo e Perosinho",
        "Silvalde",
        "Sobreira",
        "St. Tirso, Couto (S. Cristina e S. Miguel) e Burgães",
        "São Félix da Marinha",
        "São João da Madeira",
        "São Mamede de Infesta e Senhora da Hora",
        "São Martinho da Gândara",
        "São Miguel do Mato",
        "São Pedro Fins",
        "São Pedro de Castelões",
        "São Roque",
        "Touguinha e Touguinhó",
        "Tropeço",
        "Urrô",
        "Valongo",
        "Vandoma",
        "Vila Chã",
        "Vila Chã, Codal e Vila Cova de Perrinho",
        "Vila Nova da Telha",
        "Vila Nova do Campo",
        "Vila de Cucujães",
        "Vila do Conde",
        "Vilar de Andorinho",
        "Vilar de Pinheiro",
        "Vilar e Mosteiró",
        "Vilarinho",
        "Vilela",
        "Várzea",
        "Água Longa",
        "Águas Santas",
        "Árvore",
    ],
    ordered=False,
)
# Quais freguesias serão mais baratas?
print()

column_description(
    name="Latitude do imóvel",
    series=dataset.latitude,
    type_description="Real",
    description="Latitude da localização do imóvel anunciado conforme o World Geodetic System (WGS84).",
    column_type="other",
)
# Creio ser menos relevante do que a freguesia calculada.
print()

column_description(
    name="Longitude do imóvel",
    series=dataset.longitude,
    type_description="Real",
    description="Longitude da localização do imóvel anunciado conforme o World Geodetic System (WGS84).",
    column_type="other",
)
# Creio ser menos relevante do que a freguesia calculada.
print()

column_description(
    name="Tipo de locação",
    series=dataset.room_type,
    type_description="Categórico, não ordenado",
    description="Tipo da locação sendo anunciada. Especifica quais áreas do imóvel o inquilino poderá acessar.",
    column_type="categorical",
)
dataset.room_type = pd.Categorical(
    dataset.room_type,
    categories=["Entire home/apt", "Hotel room", "Private room", "Shared room"],
    ordered=False,
)
# Um imóvel menor (quarto) será mais barato?
print()

column_description(
    name="Preço de locação",
    series=dataset.price,
    type_description="Real",
    description="Preço diário de locação deste anúncio em euros.",
    column_type="numerical",
)
print()

column_description(
    name="Estadia mínima",
    series=dataset.minimum_nights,
    type_description="Inteiro",
    description="Quantidade mínima de dias que o hóspede deve contratar de estadia.",
    column_type="numerical",
)
# Um anúncio com estadia longa será mais barato?
print()

column_description(
    name="Quantidade de avaliações",
    series=dataset.number_of_reviews,
    type_description="Inteiro",
    description="Quantidade de avaliações que este anúncio recebeu por meio da plataforma Airbnb.",
    column_type="numerical",
)
# Um anúncio muito avaliado será mais barato?
print()

column_description(
    name="Data da última avaliação",
    series=dataset.last_review,
    type_description="Data (dia, mês, e ano)",
    description="Data em que a avaliação mais recente deste anúncio foi realizada por meio da plataforma Airbnb.",
    column_type="date",
)
# Creio não ter muita relevância.
print()

column_description(
    name="Média de avaliações por mês",
    series=dataset.reviews_per_month,
    type_description="Inteiro",
    description="Quantidade média de avaliações que este anúncio recebeu por mês por meio da plataforma Airbnb.",
    column_type="numerical",
)
# Um anúncio com muitas avaliações será mais barato?
print()

column_description(
    name="Quantidade de anúncios do anunciante",
    series=dataset.calculated_host_listings_count,
    type_description="Inteiro",
    description="Quantidade de anúncios que este anunciante realizou na grande região de escopo deste dataset.",
    column_type="numerical",
)
# Um anunciante que publica muitos anúncios será mais barato?
print()

column_description(
    name="Quantidade de dias disponíveis",
    series=dataset.availability_365,
    type_description="Inteiro",
    description="Quantidade de dias nos próximos 365 dias em que este anúncio estará disponível para locação.",
    column_type="numerical",
)
# Um imóvel com disponibilidade alta será mais barato?
print()

column_description(
    name="Licença",
    series=dataset.license,
    type_description="Textual",
    description="Licença de Alojamento Local (AL) obtida para operar a locação do imóvel deste anúncio.",
    column_type="other",
)
# Um imóvel sem licença registrada ou isento de licença será mais barato?
print()


=-=-=-=
Colunas
=-=-=-=

ID do anúncio
    Identificador: id
    Tipo: Textual (sequência de dígitos)
    Identificador interno dado pelo Airbnb para esta entrada.
    Vazios: 0 de 12044 (0.0%)

Nome do anúncio
    Identificador: name
    Tipo: Textual
    Nome dado pelos anunciantes a este anúncio.
    Vazios: 0 de 12044 (0.0%)

ID do anunciante
    Identificador: host_id
    Tipo: Textual (sequência de dígitos)
    Identificador interno dado pelo Airbnb para a conta do anunciante que registrou esta entrada.
    Vazios: 0 de 12044 (0.0%)

Nome do anunciante
    Identificador: host_name
    Tipo: Textual
    Nome do anunciante que registrou esta entrada. Mesmo quando há mais de uma pessoa, eles são responsáveis pela mesma conta no Airbnb.
    Vazios: 0 de 12044 (0.0%)

Município do imóvel
    Identificador: neighbourhood_group
    Tipo: Categórico, não ordenado
    Município em que o imóvel anunciado está localizado. Obtido por meio da localização geográfica.
    Vazios: 0 de 12044 (0.